# LeetCode 2237 — Count Positions on Street With Required Brightness bằng Python

Link đề: https://leetcode.com/problems/count-positions-on-street-with-required-brightness/ · Độ khó: **Medium** · Chủ đề: Array, Prefix Sum

> **Lưu ý (Premium 🔒):** đây là bài Premium; nội dung đề dưới đây được diễn đạt lại từ bản mô tả của doocs/leetcode, hãy **đối chiếu lại với đề gốc** trên LeetCode nếu bạn có quyền truy cập.

Skeleton LeetCode:

```python
class Solution:
    def meetRequirement(self, n: int, lights: List[List[int]], requirement: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Chuyển "mỗi đèn chiếu một **đoạn**" thành bài **cộng 1 trên đoạn** rồi hỏi giá trị từng điểm.
2. Thành thạo **mảng hiệu (difference array)**: cộng đoạn `O(1)`, khôi phục bằng **prefix sum** `O(n)`.
3. Xử lý đúng **cắt biên** `max(0, p − r)` và `min(n − 1, p + r)` — tránh chỉ số âm (Python không báo lỗi mà lặng lẽ đánh chỉ số từ cuối!).
4. Biết các cách tương đương: **quét sự kiện** (sweep line) và **Fenwick tree** (cập nhật đoạn, truy vấn điểm).
5. Kiểm thử bằng brute force "tô từng ô" với dữ liệu ngẫu nhiên seed cố định.

## 2. Đề bài & ví dụ

Con phố là trục số từ `0` tới `n − 1`. Mỗi `lights[i] = [position_i, range_i]` là một đèn chiếu sáng đoạn **đóng** `[max(0, position_i − range_i), min(n − 1, position_i + range_i)]`. **Độ sáng** của vị trí `p` là số đèn chiếu tới `p`. Cho mảng `requirement` độ dài `n`: `requirement[i]` là độ sáng tối thiểu cần ở vị trí `i`.

Trả về số vị trí `i ∈ [0, n − 1]` có độ sáng **≥** `requirement[i]`.

**Ví dụ 1:** `n = 5, lights = [[0,1],[2,1],[3,2]], requirement = [0,2,1,4,1]` → `4`. Các đoạn: `[0,1]`, `[1,3]`, `[1,4]` ⇒ độ sáng `[1, 3, 2, 2, 1]`; chỉ vị trí 3 (cần 4, có 2) không đạt.

**Ví dụ 2:** `n = 1, lights = [[0,1]], requirement = [2]` → `0` (độ sáng 1 < 2).

**Ràng buộc:** `1 ≤ n ≤ 10^5`, `1 ≤ len(lights) ≤ 10^5`, `0 ≤ position_i < n`, `0 ≤ range_i ≤ 10^5`, `0 ≤ requirement[i] ≤ 10^5`.

In [1]:
import random
import time
from bisect import bisect_left, bisect_right
from itertools import accumulate
from typing import List

examples = [(5, [[0, 1], [2, 1], [3, 2]], [0, 2, 1, 4, 1], 4), (1, [[0, 1]], [2], 0)]
for n, lights, req, out in examples:
    print(f"n={n}, lights={lights}, requirement={req} -> {out}")

n=5, lights=[[0, 1], [2, 1], [3, 2]], requirement=[0, 2, 1, 4, 1] -> 4
n=1, lights=[[0, 1]], requirement=[2] -> 0


## 3. Kiến thức nền

### 3.1 Đoạn sáng của một đèn

Đèn ở `p` bán kính `r` chiếu `[p − r, p + r]`, nhưng phố chỉ có `[0, n − 1]` ⇒ cắt: `lo = max(0, p − r)`, `hi = min(n − 1, p + r)`. Vì `0 ≤ p < n` nên đoạn sau cắt luôn **khác rỗng** (`lo ≤ p ≤ hi`).

In [2]:
def lamp_interval(n, p, r):
    return max(0, p - r), min(n - 1, p + r)

for p, r in [[0, 1], [2, 1], [3, 2]]:
    print(f"đèn ({p}, {r}) với n=5 -> đoạn {lamp_interval(5, p, r)}")

đèn (0, 1) với n=5 -> đoạn (0, 1)
đèn (2, 1) với n=5 -> đoạn (1, 3)
đèn (3, 2) với n=5 -> đoạn (1, 4)


### 3.2 Mảng hiệu

Muốn cộng `+1` cho mọi `i ∈ [lo, hi]`: đặt `d[lo] += 1`, `d[hi + 1] -= 1`. Sau tất cả các đoạn, `brightness[i] = d[0] + d[1] + … + d[i]` (prefix sum). Mảng `d` cần độ dài `n + 1` để `d[hi + 1]` không tràn khi `hi = n − 1`.

In [3]:
d = [0] * 6
for lo, hi in [(0, 1), (1, 3), (1, 4)]:
    d[lo] += 1; d[hi + 1] -= 1
    print(f"cộng đoạn [{lo},{hi}] -> d = {d}")
print("prefix sum:", list(accumulate(d))[:5])

cộng đoạn [0,1] -> d = [1, 0, -1, 0, 0, 0]
cộng đoạn [1,3] -> d = [1, 1, -1, 0, -1, 0]
cộng đoạn [1,4] -> d = [1, 2, -1, 0, -1, -1]
prefix sum: [1, 3, 2, 2, 1]


### 3.3 Cạm bẫy chỉ số âm trong Python

Nếu quên cắt `p − r` về 0, `d[-2] += 1` **không báo lỗi** mà sửa phần tử thứ hai từ cuối — kết quả sai lặng lẽ.

In [4]:
d = [0] * 6
d[0 - 2] += 1          # tưởng là d[-2] "ngoài phố", thực ra là d[4]
print("d sau khi ghi vào chỉ số -2:", d)

d sau khi ghi vào chỉ số -2: [0, 0, 0, 0, 1, 0]


## 4. Ý tưởng nền tảng / trực giác

Bài hỏi hai việc tách biệt:

1. **Tính độ sáng** tại mọi vị trí — đây là bài "nhiều đoạn chồng lên nhau, đếm số đoạn phủ mỗi điểm".
2. **So sánh** với `requirement` và đếm.

Bước 2 là `O(n)` hiển nhiên. Bước 1 nếu "tô" từng ô của mỗi đèn thì một đèn có thể phủ cả `n` ô ⇒ `O(n · len(lights))` = `10^10` trong trường hợp xấu. Mấu chốt: ta không cần biết **từng** đèn phủ ô nào, chỉ cần **số lượng** ⇒ mỗi đèn chỉ đóng góp hai "sự kiện": **bắt đầu** sáng ở `lo`, **hết** sáng sau `hi`.

Một cách nhìn khác: brute force tốn `Σ (hi − lo + 1)` phép cộng — tổng **độ dài** các đoạn — trong khi thông tin thật sự cần chỉ là `2m` **đầu mút**. Mọi cách nhanh (mảng hiệu, quét sự kiện, Fenwick) đều khai thác đúng điều này: độ sáng là hàm **bậc thang**, chỉ đổi giá trị tại đầu mút.

Hình dung đi bộ từ trái sang phải với một bộ đếm: gặp điểm bắt đầu thì `+1`, đi qua điểm kết thúc thì `−1`. Giá trị bộ đếm tại mỗi bước chính là độ sáng.

| Vị trí | Sự kiện | Bộ đếm (độ sáng) | Yêu cầu | Đạt? |
|---|---|---|---|---|
| 0 | đèn A bắt đầu | 1 | 0 | ✓ |
| 1 | đèn B, C bắt đầu | 3 | 2 | ✓ |
| 2 | đèn A hết (sau 1) | 2 | 1 | ✓ |
| 3 | — | 2 | 4 | ✗ |
| 4 | đèn B hết (sau 3) | 1 | 1 | ✓ |

## 5. Cách 1 — Brute Force / baseline: tô từng ô

**Ý tưởng:** với mỗi đèn, cộng 1 cho mọi ô trong đoạn của nó. Rồi đếm ô đạt yêu cầu.

In [5]:
def meet_brute(n, lights, requirement):
    bright = [0] * n
    for p, r in lights:
        for i in range(max(0, p - r), min(n - 1, p + r) + 1):
            bright[i] += 1
    return sum(b >= q for b, q in zip(bright, requirement))

for n, lights, req, out in examples:
    got = meet_brute(n, lights, req); print(n, lights, req, "->", got); assert got == out

5 [[0, 1], [2, 1], [3, 2]] [0, 2, 1, 4, 1] -> 4
1 [[0, 1]] [2] -> 0


**Độ phức tạp:** `O(n + Σ độ dài đoạn)` ≤ `O(n · m)` với `m = len(lights)`. Space `O(n)`.

**Nút thắt:** với `n = m = 10^5` và mọi đèn phủ cả phố, ~`10^10` phép cộng — quá chậm. Ta đang làm lại cùng một việc "+1 cho cả dải" bằng vòng lặp.

In [6]:
rng = random.Random(2237)
n_mid = 3000
lights_mid = [[rng.randrange(n_mid), rng.randint(n_mid // 2, n_mid)] for _ in range(3000)]
req_mid = [rng.randint(0, 3000) for _ in range(n_mid)]
t = time.perf_counter(); v = meet_brute(n_mid, lights_mid, req_mid)
print(f"brute với n = m = 3000 (đèn rất rộng): {v} trong {time.perf_counter() - t:.3f}s — n = m = 10^5 sẽ chậm hơn ~1000 lần")

brute với n = m = 3000 (đèn rất rộng): 2763 trong 0.274s — n = m = 10^5 sẽ chậm hơn ~1000 lần


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** "cộng hằng số trên một đoạn" là thao tác **cục bộ trên mảng hiệu**: mảng `bright` có hiệu kề `d[i] = bright[i] − bright[i−1]`; cộng 1 trên `[lo, hi]` chỉ làm `d[lo]` tăng 1 và `d[hi+1]` giảm 1 (các hiệu bên trong đoạn không đổi vì cả hai đầu cùng tăng). Gom mọi đèn vào `d` trong `O(m)`, rồi **một lần** prefix sum `O(n)` cho toàn bộ `bright`.

**Quy tắc "nếu… thì…":**
- **Nếu** có nhiều cập nhật "cộng trên đoạn" và chỉ hỏi **sau khi cập nhật xong** (offline) **thì** mảng hiệu.
- **Nếu** cập nhật và truy vấn **xen kẽ** (online) **thì** Fenwick tree / segment tree.
- **Nếu** tọa độ rất lớn (tới `10^9`) mà số đèn ít **thì** quét sự kiện đã sắp xếp (hoặc nén tọa độ) thay vì mảng dài `n`.

### 6.1 Vì sao cắt biên là bắt buộc với mảng hiệu?

- `p − r < 0`: phải dùng `lo = 0`. Ghi vào `d[p − r]` (âm) sẽ sửa nhầm cuối mảng (mục 3.3).
- `p + r ≥ n`: phải dùng `hi = n − 1`, ghi `d[n] −= 1`. Ghi vào `d[p + r + 1]` sẽ `IndexError` (hoặc phải cấp phát mảng dài `n + max(range)`).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mảng hiệu + prefix sum (bản nộp chính)

In [7]:
def meet_diff(n, lights, requirement):
    d = [0] * (n + 1)
    for p, r in lights:
        d[max(0, p - r)] += 1
        d[min(n - 1, p + r) + 1] -= 1
    ans = cur = 0
    for i in range(n):
        cur += d[i]                     # cur = độ sáng tại i
        if cur >= requirement[i]:
            ans += 1
    return ans

for n, lights, req, out in examples:
    assert meet_diff(n, lights, req) == out
print([meet_diff(n, l, q) for n, l, q, _ in examples])

[4, 0]


**Độ phức tạp Cách 2:** `O(n + m)` time, `O(n)` space.

### 7.2 Cách 3 — Quét sự kiện đã sắp xếp (sweep line)

Tạo danh sách điểm bắt đầu `starts` và điểm "hết sáng" `ends = hi + 1`, sắp xếp. Độ sáng tại `i` = (số `start ≤ i`) − (số `end ≤ i`) — tính bằng `bisect_right` hoặc hai con trỏ. Cách này không cần mảng độ dài `n` nếu chỉ hỏi ít vị trí.

In [8]:
def meet_sweep(n, lights, requirement):
    starts = sorted(max(0, p - r) for p, r in lights)
    ends = sorted(min(n - 1, p + r) + 1 for p, r in lights)
    ans = 0; s = e = 0
    for i in range(n):
        while s < len(starts) and starts[s] <= i: s += 1      # đèn đã bật
        while e < len(ends) and ends[e] <= i: e += 1          # đèn đã tắt
        if s - e >= requirement[i]:
            ans += 1
    return ans

def meet_bisect(n, lights, requirement):
    starts = sorted(max(0, p - r) for p, r in lights)
    ends = sorted(min(n - 1, p + r) for p, r in lights)
    # độ sáng(i) = #{lo <= i} - #{hi < i}
    return sum(bisect_right(starts, i) - bisect_left(ends, i) >= requirement[i] for i in range(n))

for n, lights, req, out in examples:
    assert meet_sweep(n, lights, req) == out == meet_bisect(n, lights, req)
print([meet_sweep(n, l, q) for n, l, q, _ in examples], [meet_bisect(n, l, q) for n, l, q, _ in examples])

[4, 0] [4, 0]


**Độ phức tạp Cách 3:** sắp xếp `O(m log m)` + quét `O(n + m)` (hai con trỏ) hoặc `O(n log m)` (bisect). Space `O(m)`.

### 7.3 Cách 4 — Fenwick tree (cập nhật đoạn, truy vấn điểm)

Fenwick lưu mảng hiệu `d`; `add(lo, +1)`, `add(hi+1, −1)`, và `query(i)` = prefix sum = độ sáng tại `i`. Thừa thãi cho bài offline này, nhưng là bước đệm cho biến thể online (đèn bật/tắt xen kẽ câu hỏi).

In [9]:
class Fenwick:
    def __init__(self, n):
        self.n = n; self.t = [0] * (n + 1)
    def add(self, i, v):                 # i: 0-indexed
        i += 1
        while i <= self.n:
            self.t[i] += v; i += i & -i
    def query(self, i):                  # tổng d[0..i]
        i += 1; s = 0
        while i > 0:
            s += self.t[i]; i -= i & -i
        return s

def meet_fenwick(n, lights, requirement):
    fw = Fenwick(n + 1)
    for p, r in lights:
        fw.add(max(0, p - r), 1); fw.add(min(n - 1, p + r) + 1, -1)
    return sum(fw.query(i) >= requirement[i] for i in range(n))

for n, lights, req, out in examples:
    assert meet_fenwick(n, lights, req) == out
print([meet_fenwick(n, l, q) for n, l, q, _ in examples])

[4, 0]


**Độ phức tạp Cách 4:** `O((n + m) log n)` time, `O(n)` space.

### 7.4 Cách 5 — Mảng hiệu viết gọn với `accumulate`

In [10]:
def meet_accumulate(n, lights, requirement):
    d = [0] * (n + 1)
    for p, r in lights:
        d[max(0, p - r)] += 1; d[min(n - 1, p + r) + 1] -= 1
    return sum(b >= q for b, q in zip(accumulate(d), requirement))   # zip dừng ở n phần tử

for n, lights, req, out in examples:
    assert meet_accumulate(n, lights, req) == out
print([meet_accumulate(n, l, q) for n, l, q, _ in examples])

[4, 0]


## 8. TRACE từng bước

Trace Cách 2 trên ví dụ 1: `n = 5`, `lights = [[0,1],[2,1],[3,2]]`, `requirement = [0,2,1,4,1]`.

**Pha 1 — ghi mảng hiệu:**

| Đèn | Đoạn | `d` sau khi ghi |
|---|---|---|
| (0, 1) | [0, 1] | [1, 0, −1, 0, 0, 0] |
| (2, 1) | [1, 3] | [1, 1, −1, 0, −1, 0] |
| (3, 2) | [1, 4] | [1, 2, −1, 0, −1, −1] |

**Pha 2 — prefix sum và so sánh:**

| i | d[i] | cur | req[i] | đạt? | ans |
|---|---|---|---|---|---|
| 0 | 1 | 1 | 0 | ✓ | 1 |
| 1 | 2 | 3 | 2 | ✓ | 2 |
| 2 | −1 | 2 | 1 | ✓ | 3 |
| 3 | 0 | 2 | 4 | ✗ | 3 |
| 4 | −1 | 1 | 1 | ✓ | 4 |

In [11]:
def trace_diff(n, lights, requirement):
    d = [0] * (n + 1)
    for p, r in lights:
        lo, hi = max(0, p - r), min(n - 1, p + r)
        d[lo] += 1; d[hi + 1] -= 1
        print(f"đèn ({p},{r}) -> đoạn [{lo},{hi}] | d = {d}")
    ans = cur = 0
    for i in range(n):
        cur += d[i]; ok = cur >= requirement[i]; ans += ok
        print(f"i={i}: d[i]={d[i]:2d}, độ sáng={cur}, cần={requirement[i]}, {'đạt' if ok else 'KHÔNG đạt'}, ans={ans}")
    return ans

assert trace_diff(5, [[0, 1], [2, 1], [3, 2]], [0, 2, 1, 4, 1]) == 4

đèn (0,1) -> đoạn [0,1] | d = [1, 0, -1, 0, 0, 0]
đèn (2,1) -> đoạn [1,3] | d = [1, 1, -1, 0, -1, 0]
đèn (3,2) -> đoạn [1,4] | d = [1, 2, -1, 0, -1, -1]
i=0: d[i]= 1, độ sáng=1, cần=0, đạt, ans=1
i=1: d[i]= 2, độ sáng=3, cần=2, đạt, ans=2
i=2: d[i]=-1, độ sáng=2, cần=1, đạt, ans=3
i=3: d[i]= 0, độ sáng=2, cần=4, KHÔNG đạt, ans=3
i=4: d[i]=-1, độ sáng=1, cần=1, đạt, ans=4


Trace thêm một ví dụ **tự dựng** (đã kiểm bằng brute force) có đèn tràn cả hai biên: `n = 6`, `lights = [[1, 4], [5, 0], [4, 10]]`, `requirement = [2, 1, 3, 2, 2, 3]`.

In [12]:
own = (6, [[1, 4], [5, 0], [4, 10]], [2, 1, 3, 2, 2, 3])
got = trace_diff(*own)
print("brute:", meet_brute(*own)); assert got == meet_brute(*own)

đèn (1,4) -> đoạn [0,5] | d = [1, 0, 0, 0, 0, 0, -1]
đèn (5,0) -> đoạn [5,5] | d = [1, 0, 0, 0, 0, 1, -2]
đèn (4,10) -> đoạn [0,5] | d = [2, 0, 0, 0, 0, 1, -3]
i=0: d[i]= 2, độ sáng=2, cần=2, đạt, ans=1
i=1: d[i]= 0, độ sáng=2, cần=1, đạt, ans=2
i=2: d[i]= 0, độ sáng=2, cần=3, KHÔNG đạt, ans=2
i=3: d[i]= 0, độ sáng=2, cần=2, đạt, ans=3
i=4: d[i]= 0, độ sáng=2, cần=2, đạt, ans=4
i=5: d[i]= 1, độ sáng=3, cần=3, đạt, ans=5
brute: 5


### 8.1 Trace Cách 3 (quét sự kiện hai con trỏ) trên ví dụ 1

`starts = [0, 1, 1]` (đã sắp), `ends = [2, 4, 5]` (= `hi + 1`, đã sắp). Tại mỗi `i`: tiến con trỏ `s` qua các `start ≤ i`, con trỏ `e` qua các `end ≤ i`; độ sáng = `s − e` = số đèn đã bật trừ số đèn đã tắt.

In [13]:
def trace_sweep(n, lights, requirement):
    starts = sorted(max(0, p - r) for p, r in lights)
    ends = sorted(min(n - 1, p + r) + 1 for p, r in lights)
    print("starts =", starts, "| ends (hi+1) =", ends)
    s = e = ans = 0
    for i in range(n):
        while s < len(starts) and starts[s] <= i: s += 1
        while e < len(ends) and ends[e] <= i: e += 1
        ok = s - e >= requirement[i]; ans += ok
        print(f"i={i}: đã bật s={s}, đã tắt e={e}, độ sáng={s - e}, cần={requirement[i]} -> {'đạt' if ok else 'không'}")
    return ans

assert trace_sweep(5, [[0, 1], [2, 1], [3, 2]], [0, 2, 1, 4, 1]) == 4

starts = [0, 1, 1] | ends (hi+1) = [2, 4, 5]
i=0: đã bật s=1, đã tắt e=0, độ sáng=1, cần=0 -> đạt
i=1: đã bật s=3, đã tắt e=0, độ sáng=3, cần=2 -> đạt
i=2: đã bật s=3, đã tắt e=1, độ sáng=2, cần=1 -> đạt
i=3: đã bật s=3, đã tắt e=1, độ sáng=2, cần=4 -> không
i=4: đã bật s=3, đã tắt e=2, độ sáng=1, cần=1 -> đạt


**Nhận xét:** hai con trỏ chỉ tiến, tổng số bước tiến là `2m` ⇒ phần quét `O(n + m)`; chi phí chính là sắp xếp. Khi `n` rất lớn (tọa độ tới `10^9`) ta không duyệt từng `i` mà chỉ duyệt các **tọa độ sự kiện** — độ sáng chỉ thay đổi tại đó (xem biến thể 1 ở mục 15).

## 9. Code nộp LeetCode

In [14]:
class Solution:
    def meetRequirement(self, n: int, lights: List[List[int]], requirement: List[int]) -> int:
        d = [0] * (n + 1)                       # mảng hiệu, thêm 1 ô cho hi + 1 = n
        for p, r in lights:
            d[max(0, p - r)] += 1               # bắt đầu sáng (cắt biên trái)
            d[min(n - 1, p + r) + 1] -= 1       # hết sáng sau hi (cắt biên phải)
        ans = cur = 0
        for i in range(n):
            cur += d[i]                         # độ sáng tại i
            if cur >= requirement[i]:
                ans += 1
        return ans

for n, lights, req, out in examples:
    assert Solution().meetRequirement(n, lights, req) == out
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Bất biến của mảng hiệu.** Gọi `cover_j(i) = 1` nếu đèn `j` chiếu tới `i` (tức `lo_j ≤ i ≤ hi_j`), ngược lại 0. Đèn `j` đóng góp vào `d` đúng hai giá trị: `+1` tại `lo_j`, `−1` tại `hi_j + 1`. Prefix sum tới `i` của phần đóng góp này là:
- `0` nếu `i < lo_j` (chưa gặp cả hai),
- `1` nếu `lo_j ≤ i ≤ hi_j` (đã gặp `+1`, chưa gặp `−1`),
- `0` nếu `i > hi_j` (đã gặp cả hai).

Tức đúng bằng `cover_j(i)`. Prefix sum tuyến tính ⇒ `cur` tại `i` = `Σ_j cover_j(i)` = độ sáng tại `i`.

**Cắt biên.** Đoạn sáng thực tế trên phố là `[max(0, p−r), min(n−1, p+r)]` theo đúng định nghĩa đề; vì `0 ≤ p ≤ n−1` nên `lo ≤ hi`, và `hi + 1 ≤ n` nằm trong mảng `d` độ dài `n + 1`.

**Quét sự kiện (Cách 3) đúng vì cùng lý do.** `#{lo_j ≤ i} − #{hi_j + 1 ≤ i}` = số đèn đã bắt đầu trừ số đèn đã kết thúc trước hoặc tại `i`; mọi đèn đã kết thúc thì cũng đã bắt đầu (`lo_j ≤ hi_j < hi_j + 1`), nên hiệu đúng bằng số đèn đang sáng tại `i`.

**Đếm.** Duyệt mọi `i ∈ [0, n−1]` và cộng 1 khi `cur ≥ requirement[i]` — đúng định nghĩa đáp án. Bộ test so với brute force tô từng ô.

## 11. Phân tích Time / Space

Gọi `m = len(lights)`.

| Cách | Time | Space |
|---|---|---|
| 1. Tô từng ô | `O(n + Σ len)` ≤ `O(n·m)` | `O(n)` |
| 2. Mảng hiệu | `O(n + m)` | `O(n)` |
| 3. Quét sự kiện | `O(m log m + n)` (hoặc `+ n log m` với bisect) | `O(m)` |
| 4. Fenwick | `O((n + m) log n)` | `O(n)` |
| 5. `accumulate` | `O(n + m)` | `O(n)` |

Đo với `n = m = 10^5`:

In [15]:
rng = random.Random(1)
n_big = 10**5
lights_big = [[rng.randrange(n_big), rng.randint(0, 10**5)] for _ in range(10**5)]
req_big = [rng.randint(0, 10**5) for _ in range(n_big)]
for name, fn in [("diff", meet_diff), ("sweep", meet_sweep), ("bisect", meet_bisect), ("fenwick", meet_fenwick), ("accumulate", meet_accumulate)]:
    t = time.perf_counter(); v = fn(n_big, lights_big, req_big)
    print(f"{name:10s}: {v} trong {time.perf_counter() - t:.3f}s")

diff      : 66692 trong 0.023s


sweep     : 66692 trong 0.078s


bisect    : 66692 trong 0.078s


fenwick   : 66692 trong 0.231s
accumulate: 66692 trong 0.021s


## 12. Bộ test

So mọi cách với brute force: ví dụ đề, biên (một vị trí, bán kính 0, bán kính rất lớn, yêu cầu 0 ở mọi nơi, nhiều đèn cùng vị trí), và ngẫu nhiên seed cố định.

In [16]:
fns = {"diff": meet_diff, "sweep": meet_sweep, "bisect": meet_bisect, "fenwick": meet_fenwick,
       "accumulate": meet_accumulate, "Solution": lambda n, l, q: Solution().meetRequirement(n, l, q)}
edge = [(1, [[0, 0]], [1], 1), (1, [[0, 100000]], [0], 1), (3, [[1, 0]], [0, 1, 0], 3), (3, [[1, 0]], [1, 1, 1], 1),
        (4, [[0, 10], [3, 10]], [2, 2, 2, 3], 3), (5, [[2, 0]] * 4, [0, 0, 4, 0, 0], 5), (5, [[2, 0]] * 4, [0, 0, 5, 0, 0], 4), own + (None,)]
for n, lights, req, out in examples + edge:
    exp = meet_brute(n, lights, req)
    if out is not None: assert exp == out, (n, lights, req, exp)
    for name, fn in fns.items():
        assert fn(n, lights, req) == exp, (name, n, lights, req)
print("Ví dụ + biên: OK")

Ví dụ + biên: OK


In [17]:
rng = random.Random(42)
for _ in range(500):
    n = rng.randint(1, 30); m = rng.randint(1, 12)
    lights = [[rng.randrange(n), rng.randint(0, 2 * n)] for _ in range(m)]
    req = [rng.randint(0, m) for _ in range(n)]
    exp = meet_brute(n, lights, req)
    for name, fn in fns.items():
        assert fn(n, lights, req) == exp, (name, n, lights, req)
print("500 test ngẫu nhiên: mọi cách khớp brute force")

500 test ngẫu nhiên: mọi cách khớp brute force


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Quên cắt biên trái (`p − r` âm)

Bản sai ghi `d[p − r]` với chỉ số âm. Ở ví dụ 1, đèn `(0, 1)` ghi vào `d[−1]` = `d[5]` — ô cuối không bao giờ được cộng dồn — nên đèn đó **mất hẳn** phần `+1`, trong khi phần `−1` ở `d[2]` vẫn còn ⇒ độ sáng các ô sau bị hụt. Python không báo lỗi gì. Ta cũng tìm vét cạn phản ví dụ nhỏ nhất.

In [18]:
def wrong_no_left_clip(n, lights, requirement):
    d = [0] * (n + 1)
    for p, r in lights:
        d[p - r] += 1                                            # SAI: chỉ số âm -> sửa cuối mảng
        d[min(n - 1, p + r) + 1] -= 1
    cur = ans = 0
    for i in range(n):
        cur += d[i]; ans += cur >= requirement[i]
    return ans

bad = examples[0][:3]
print("ví dụ 1: sai =", wrong_no_left_clip(*bad), "| đúng =", meet_brute(*bad), "(d[-1] là d[n]: đèn (0,1) mất phần +1)")

from itertools import product as iproduct
def tiny_cases():
    """Mọi bộ nhỏ: n <= 3, một đèn, bán kính <= 2, yêu cầu 0..2 mỗi ô."""
    for n in range(1, 4):
        for p in range(n):
            for r in range(3):
                for req in iproduct(range(3), repeat=n):
                    yield n, [[p, r]], list(req)

bad1 = next(c for c in tiny_cases() if wrong_no_left_clip(*c) != meet_brute(*c))
print("phản ví dụ nhỏ nhất:", bad1, "| sai =", wrong_no_left_clip(*bad1), "| đúng =", meet_brute(*bad1))

ví dụ 1: sai = 3 | đúng = 4 (d[-1] là d[n]: đèn (0,1) mất phần +1)
phản ví dụ nhỏ nhất: (1, [[0, 1]], [1]) | sai = 0 | đúng = 1


In [19]:
print("sửa (max(0, p - r)):", meet_diff(*bad1), "| ví dụ 1:", meet_diff(*bad))

sửa (max(0, p - r)): 1 | ví dụ 1: 4


### Lỗi 2 — Quên cắt biên phải (`p + r + 1 > n`)

Bản sai ghi `d[p + r + 1]`, dễ ném `IndexError`. Ta bắt lỗi để xem thông báo.

In [20]:
def wrong_no_right_clip(n, lights, requirement):
    d = [0] * (n + 1)
    try:
        for p, r in lights:
            d[max(0, p - r)] += 1
            d[p + r + 1] -= 1                                    # SAI: có thể vượt n
    except IndexError as err:
        return f"IndexError: {err}"
    cur = ans = 0
    for i in range(n):
        cur += d[i]; ans += cur >= requirement[i]
    return ans

print("ví dụ 1: sai ->", wrong_no_right_clip(*bad))

ví dụ 1: sai -> IndexError: list index out of range


In [21]:
print("sửa (min(n - 1, p + r) + 1):", meet_diff(*bad))

sửa (min(n - 1, p + r) + 1): 4


### Lỗi 3 — Coi đoạn là **nửa mở** (`d[hi] -= 1` thay vì `d[hi + 1]`)

Đoạn sáng là **đóng** `[lo, hi]`. Trừ ở `hi` làm vị trí `hi` mất ánh sáng của đèn đó. Tìm tự động phản ví dụ nhỏ nhất.

In [22]:
def wrong_half_open(n, lights, requirement):
    d = [0] * (n + 1)
    for p, r in lights:
        d[max(0, p - r)] += 1
        d[min(n - 1, p + r)] -= 1                                # SAI: phải là hi + 1
    cur = ans = 0
    for i in range(n):
        cur += d[i]; ans += cur >= requirement[i]
    return ans

bad3 = next(c for c in tiny_cases() if wrong_half_open(*c) != meet_brute(*c))
print("phản ví dụ:", bad3, "| sai =", wrong_half_open(*bad3), "| đúng =", meet_brute(*bad3))

phản ví dụ: (1, [[0, 0]], [1]) | sai = 0 | đúng = 1


In [23]:
print("sửa:", meet_diff(*bad3))

sửa: 1


### Lỗi 4 — So sánh `>` thay vì `≥`

Đề hỏi độ sáng **ít nhất** bằng yêu cầu. Vị trí 4 ở ví dụ 1 (độ sáng 1 = yêu cầu 1) bị loại sai.

In [24]:
def wrong_strict(n, lights, requirement):
    d = [0] * (n + 1)
    for p, r in lights:
        d[max(0, p - r)] += 1; d[min(n - 1, p + r) + 1] -= 1
    cur = ans = 0
    for i in range(n):
        cur += d[i]; ans += cur > requirement[i]                 # SAI: phải là >=
    return ans

print("ví dụ 1: sai =", wrong_strict(*bad), "| đúng =", meet_diff(*bad))

ví dụ 1: sai = 3 | đúng = 4


### Lỗi 5 — Đọc `d[i]` (hiệu) như độ sáng, quên prefix sum

In [25]:
def wrong_no_prefix(n, lights, requirement):
    d = [0] * (n + 1)
    for p, r in lights:
        d[max(0, p - r)] += 1; d[min(n - 1, p + r) + 1] -= 1
    return sum(d[i] >= requirement[i] for i in range(n))          # SAI: d[i] không phải độ sáng

print("ví dụ 1: sai =", wrong_no_prefix(*bad), "| đúng =", meet_diff(*bad))

ví dụ 1: sai = 2 | đúng = 4


In [26]:
print("sửa (cộng dồn):", meet_accumulate(*bad))

sửa (cộng dồn): 4


### Lỗi 6 — Quét sự kiện nhưng sắp xếp **chung** start/end rồi xử lý sai thứ tự tại cùng một điểm

Gộp sự kiện `(x, +1)` và `(hi, −1)` (dùng `hi` thay vì `hi + 1`) rồi sắp xếp: tại cùng tọa độ `−1` đứng trước `+1` (vì `−1 < +1`) nên đèn bị tắt **trước khi** vị trí `hi` được tính.

In [27]:
def wrong_event_order(n, lights, requirement):
    ev = sorted([(max(0, p - r), 1) for p, r in lights] + [(min(n - 1, p + r), -1) for p, r in lights])   # SAI
    ans = cur = k = 0
    for i in range(n):
        while k < len(ev) and ev[k][0] <= i:
            cur += ev[k][1]; k += 1
        ans += cur >= requirement[i]
    return ans

bad6 = next(c for c in tiny_cases() if wrong_event_order(*c) != meet_brute(*c))
print("phản ví dụ:", bad6, "| sai =", wrong_event_order(*bad6), "| đúng =", meet_brute(*bad6))
print("sửa (sự kiện kết thúc tại hi + 1):", meet_sweep(*bad6))

phản ví dụ: (1, [[0, 0]], [1]) | sai = 0 | đúng = 1
sửa (sự kiện kết thúc tại hi + 1): 1


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Nhiều đoạn, hỏi mỗi điểm bị phủ bao nhiêu lần" ⇒ **mảng hiệu** (`+1` ở `lo`, `−1` ở `hi + 1`) + prefix sum.
- Luôn hỏi: đoạn **đóng** hay **nửa mở**? Biên có cần **cắt** không?
- Cấp phát `d` dài `n + 1` (hoặc `n + 2`) để khỏi phải `if hi + 1 < n`.
- Python: chỉ số âm **không** báo lỗi — cắt biên trước khi ghi.
- Tọa độ lớn, ít đoạn ⇒ sự kiện sắp xếp / nén tọa độ thay vì mảng dài.

## 15. Biến thể của bài

1. **LeetCode 2021 — Brightest Position on Street:** tọa độ tới `10^8`, tìm vị trí sáng nhất ⇒ quét sự kiện đã sắp xếp (không dùng mảng dài).
2. **LeetCode 1094 — Car Pooling / 1109 — Corporate Flight Bookings:** cộng giá trị trên đoạn bằng mảng hiệu.
3. **LeetCode 2381 — Shifting Letters II:** mảng hiệu trên chữ cái (mod 26).
4. **Online:** đèn bật/tắt xen kẽ câu hỏi "độ sáng tại `p`" ⇒ Fenwick (Cách 4).
5. **Độ sáng có trọng số:** mỗi đèn có công suất `w` ⇒ `d[lo] += w`, `d[hi+1] −= w`.
6. **Hai chiều:** đèn chiếu hình chữ nhật ⇒ mảng hiệu 2D (bài tập 5).

Minh họa biến thể 1 (LeetCode 2021: trả vị trí **nhỏ nhất** có độ sáng lớn nhất; các đoạn `[p − r, p + r]` không cắt biên):

In [28]:
def brightest_position(lights):
    ev = sorted([(p - r, 1) for p, r in lights] + [(p + r + 1, -1) for p, r in lights])
    best = cur = 0; pos = None
    for x, v in ev:                     # tại cùng x: -1 đứng trước +1 (đúng vì -1 là tắt từ x)
        cur += v
        if cur > best: best, pos = cur, x
    return pos

def brightest_brute(lights):
    lo = min(p - r for p, r in lights); hi = max(p + r for p, r in lights)
    cnt = {x: sum(p - r <= x <= p + r for p, r in lights) for x in range(lo, hi + 1)}
    m = max(cnt.values()); return min(x for x, c in cnt.items() if c == m)

rngb = random.Random(5)
for _ in range(300):
    L = [[rngb.randint(-10, 10), rngb.randint(0, 5)] for _ in range(rngb.randint(1, 6))]
    assert brightest_position(L) == brightest_brute(L), L
print("biến thể 1 khớp brute; [[-3,2],[1,2],[-1,2]] ->", brightest_position([[-3, 2], [1, 2], [-1, 2]]))

biến thể 1 khớp brute; [[-3,2],[1,2],[-1,2]] -> -1


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Tô từng ô | cộng 1 từng ô | `O(n·m)` | `O(n)` | kiểm thử |
| 2. Mảng hiệu | `+1` / `−1` + prefix | `O(n + m)` | `O(n)` | **nộp bài** |
| 3. Quét sự kiện | đếm start/end đã qua | `O(m log m + n)` | `O(m)` | tọa độ lớn |
| 4. Fenwick | mảng hiệu động | `O((n+m) log n)` | `O(n)` | cập nhật online |
| 5. `accumulate` | như 2, gọn | `O(n + m)` | `O(n)` | viết nhanh |

## 17. Bài tập (từ dễ đến khó)

1. **Đoán output:** `n = 3, lights = [[1, 1]], requirement = [1, 1, 2]`.
2. **Tính tay:** mảng `d` và độ sáng cho `n = 4, lights = [[0, 0], [3, 5], [2, 1]]`.
3. **Điền TODO:** `brightness(n, lights)` trả về mảng độ sáng (dài `n`) bằng mảng hiệu.
4. **Tự viết lại:** viết lại Cách 2 không nhìn code.
5. **Mini challenge:** phố 2D `R × C`, mỗi đèn `[r, c, d]` chiếu hình vuông `[r−d, r+d] × [c−d, c+d]` (cắt biên); đếm ô có độ sáng `≥ req[r][c]`. Dùng mảng hiệu 2D, kiểm bằng brute force.

In [29]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def brightness(n, lights):
    # TODO
    pass

In [30]:
# Bài 4 — TODO (chỉ định nghĩa, không gọi)
def my_meet_requirement(n, lights, requirement):
    # TODO
    pass

In [31]:
# Bài 5 — TODO (chỉ định nghĩa, không gọi)
def meet_requirement_2d(R, C, lamps, req):
    # TODO
    pass

## 18. Gợi ý 3 mức

**Bài 1.** (1) Đoạn `[0, 2]`. (2) Độ sáng `[1, 1, 1]`. (3) Đạt ở vị trí 0 và 1 ⇒ `2`.

**Bài 2.** (1) Đoạn sau cắt biên: `[0,0]`, `[0,3]` (vì `3 + 5` vượt phố), `[1,3]`. (2) Ghi lần lượt: `d[0]+1, d[1]−1`; `d[0]+1, d[4]−1`; `d[1]+1, d[4]−1` ⇒ `d = [2, 0, 0, 0, −2]` (hai thao tác ở `d[1]` triệt tiêu). (3) Prefix sum ⇒ độ sáng `[2, 2, 2, 2]`.

**Bài 3.** (1) `d = [0]*(n+1)`. (2) Ghi `+1`/`−1` có cắt biên. (3) `list(accumulate(d))[:n]`.

**Bài 4.** (1) Như Bài 3. (2) Đếm `b >= q`. (3) Kiểm bằng brute.

**Bài 5.** (1) 2D: `D[r1][c1] += 1`, `D[r1][c2+1] −= 1`, `D[r2+1][c1] −= 1`, `D[r2+1][c2+1] += 1`. (2) Prefix sum theo hàng rồi theo cột. (3) Mảng `D` cỡ `(R+1) × (C+1)`.

## 19. Đáp án tham khảo

In [32]:
# Bài 1, 2
assert Solution().meetRequirement(3, [[1, 1]], [1, 1, 2]) == 2
trace_diff(4, [[0, 0], [3, 5], [2, 1]], [0, 0, 0, 0])
print("Bài 1: 2")

đèn (0,0) -> đoạn [0,0] | d = [1, -1, 0, 0, 0]
đèn (3,5) -> đoạn [0,3] | d = [2, -1, 0, 0, -1]
đèn (2,1) -> đoạn [1,3] | d = [2, 0, 0, 0, -2]
i=0: d[i]= 2, độ sáng=2, cần=0, đạt, ans=1
i=1: d[i]= 0, độ sáng=2, cần=0, đạt, ans=2
i=2: d[i]= 0, độ sáng=2, cần=0, đạt, ans=3
i=3: d[i]= 0, độ sáng=2, cần=0, đạt, ans=4
Bài 1: 2


In [33]:
# Bài 3, 4
def brightness_ans(n, lights):
    d = [0] * (n + 1)
    for p, r in lights:
        d[max(0, p - r)] += 1; d[min(n - 1, p + r) + 1] -= 1
    return list(accumulate(d))[:n]

def brightness_brute(n, lights):
    return [sum(max(0, p - r) <= i <= min(n - 1, p + r) for p, r in lights) for i in range(n)]

my_meet_requirement_ans = meet_diff
rng3 = random.Random(8)
for _ in range(300):
    n = rng3.randint(1, 20); lights = [[rng3.randrange(n), rng3.randint(0, 25)] for _ in range(rng3.randint(1, 8))]
    req = [rng3.randint(0, 8) for _ in range(n)]
    assert brightness_ans(n, lights) == brightness_brute(n, lights)
    assert my_meet_requirement_ans(n, lights, req) == meet_brute(n, lights, req)
print("Bài 3, 4: OK; brightness(4, [[0,0],[3,5],[2,1]]) =", brightness_ans(4, [[0, 0], [3, 5], [2, 1]]))

Bài 3, 4: OK; brightness(4, [[0,0],[3,5],[2,1]]) = [2, 2, 2, 2]


In [34]:
# Bài 5
def meet_requirement_2d_ans(R, C, lamps, req):
    D = [[0] * (C + 1) for _ in range(R + 1)]
    for r, c, d in lamps:
        r1, r2, c1, c2 = max(0, r - d), min(R - 1, r + d), max(0, c - d), min(C - 1, c + d)
        D[r1][c1] += 1; D[r1][c2 + 1] -= 1; D[r2 + 1][c1] -= 1; D[r2 + 1][c2 + 1] += 1
    for i in range(R + 1):
        for j in range(1, C + 1): D[i][j] += D[i][j - 1]
    for i in range(1, R + 1):
        for j in range(C + 1): D[i][j] += D[i - 1][j]
    return sum(D[i][j] >= req[i][j] for i in range(R) for j in range(C))

def meet_requirement_2d_brute(R, C, lamps, req):
    return sum(sum(abs(i - r) <= d and abs(j - c) <= d for r, c, d in lamps) >= req[i][j] for i in range(R) for j in range(C))

rng4 = random.Random(10)
for _ in range(200):
    R, Cc = rng4.randint(1, 6), rng4.randint(1, 6)
    lamps = [[rng4.randrange(R), rng4.randrange(Cc), rng4.randint(0, 4)] for _ in range(rng4.randint(1, 5))]
    req = [[rng4.randint(0, 4) for _ in range(Cc)] for _ in range(R)]
    assert meet_requirement_2d_ans(R, Cc, lamps, req) == meet_requirement_2d_brute(R, Cc, lamps, req)
print("Bài 5: OK")

Bài 5: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template mảng hiệu:**

```python
d = [0] * (n + 1)
for lo, hi in intervals:          # đoạn đóng, đã cắt biên
    d[lo] += 1
    d[hi + 1] -= 1
cur = 0
for i in range(n):
    cur += d[i]                   # giá trị tại i
```

In [35]:
def template(n, intervals):
    d = [0] * (n + 1)
    for lo, hi in intervals:
        d[lo] += 1; d[hi + 1] -= 1
    out, cur = [], 0
    for i in range(n):
        cur += d[i]; out.append(cur)
    return out
print(template(5, [(0, 1), (1, 3), (1, 4)]))

[1, 3, 2, 2, 1]


**Checklist:**

- [ ] Cắt biên: `lo = max(0, p − r)`, `hi = min(n − 1, p + r)`.
- [ ] Đoạn **đóng** ⇒ trừ ở `hi + 1`; mảng `d` dài `n + 1`.
- [ ] Prefix sum trước khi so sánh (không đọc `d[i]` trực tiếp).
- [ ] So sánh `≥` (ít nhất), không phải `>`.
- [ ] Biên: `n = 1`, bán kính 0, bán kính khổng lồ, nhiều đèn cùng chỗ.